In [71]:
import pandas as pd
import numpy as np
import lightgbm as lgb

RAW = '../data/raw/'
train = pd.read_csv(RAW + 'train.csv')
test  = pd.read_csv(RAW + 'test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])

def rmsle(gercek, tahmin):
    return np.sqrt(np.mean((np.log1p(tahmin) - np.log1p(gercek))**2))

print('train', train.shape, '| test', test.shape)

train (1226237, 5) | test (714688, 5)


EDA bulgusu: 444 satır fiziksel olarak imkansız (guc x 24 saat) teorik imkansız,
çoğunluğu sadece 4 trafoda, toplam 23 trafoda yoğun. Sorun lokalize olduğu icin global clipping
yerine trafo bazlı medyan ile düzelttik

In [72]:
train['max_teorik']  = train['guc'] * 24
train['asiri_oran']  = train['tuketim'] / train['max_teorik']
train['tuketim_duzeltilmis'] = train['tuketim']

problemli = train.loc[train['asiri_oran'] > 2, 'tanim'].unique()
for t in problemli:
    medyan = train.loc[(train['tanim']==t) & (train['asiri_oran']<=2), 'tuketim'].median()
    train.loc[(train['tanim']==t) & (train['asiri_oran']>2), 'tuketim_duzeltilmis'] = medyan

print('Düzeltilen satır:', (train['tuketim'] != train['tuketim_duzeltilmis']).sum(), '| problemli trafo:', len(problemli))

Düzeltilen satır: 444 | problemli trafo: 23


In [73]:
TATILLER = pd.to_datetime([
    # 2025
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23',
    '2025-05-01','2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09',
    '2025-08-30','2025-10-29',
    # 2026 (test dönemi dahil)
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30','2026-08-30','2026-10-29',
]).date

GUC_BINS = [0, 100, 250, 500, 1000, 2000, 5000, np.inf]

def temel_ozellikler(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il']    = lok[0]
    df['bolge'] = lok[1]
    df['ilce']  = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')

    df['ay']            = df['tarih'].dt.month
    df['gun']           = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu']    = (df['haftanin_gunu'] >= 5).astype(int)
    df['yilin_gunu']    = df['tarih'].dt.dayofyear
    df['tatil']         = df['tarih'].dt.date.isin(TATILLER).astype(int)

    df['guc_bucket']     = pd.cut(df['guc'], bins=GUC_BINS)
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    return df

train = temel_ozellikler(train)
test  = temel_ozellikler(test)

KATEGORIK = ['il','bolge','ilce','guc_bucket_str']
print('train ->', train.shape, '| test ->', test.shape)
print('tatil işaretli satır -> train:', train['tatil'].sum(), '| test ->', test['tatil'].sum())

train -> (1226237, 19) | test -> (714688, 16)
tatil işaretli satır -> train: 48245 | test -> 48256


In [74]:
#trafo profili log uzayda

def trafo_profili_olustur(gecmis, origin_tarihi):
    g = gecmis[gecmis['tarih'] < origin_tarihi].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])

    profil = g.groupby('tanim').agg(
        trafo_ortalama   = ('tuketim_duzeltilmis','mean'),
        trafo_medyan     = ('tuketim_duzeltilmis','median'),
        trafo_gun_sayisi = ('tuketim_duzeltilmis','count'),
        log_ortalama     = ('log_t','mean'),
        log_medyan       = ('log_t','median'),
        log_std          = ('log_t','std'),
        sifir_orani      = ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
    ).reset_index()

    for pencere in [30, 90]:
        son  = g[g['tarih'] >= origin_tarihi - pd.Timedelta(days=pencere)]
        ozet = son.groupby('tanim').agg(**{
            f'son{pencere}_log_ort': ('log_t','mean'),
            f'son{pencere}_sifir'  : ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
        }).reset_index()
        profil = profil.merge(ozet, on='tanim', how='left')

    profil['log_trend'] = profil['son30_log_ort'] - profil['son90_log_ort']
    return profil


def trafo_gun_profili(gecmis, origin_tarihi):
    g = gecmis[gecmis['tarih'] < origin_tarihi][['tanim','haftanin_gunu','tuketim_duzeltilmis']].copy()
    ort = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_ort')
    g = g.merge(ort, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_ort'].replace(0, np.nan)
    return g.groupby(['tanim','haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()

veri_seti_kur

Tek fonksiyon, 3 yerde aynı şekilde kullanılır (egitim / validation / test) —> sadece
origin_tarihi değişir. Bu, egitim ve tahmin koşullarının birebir aynı olmasını garantiler.

- Grup istatistiği shrinkage'li (k=30): küçükk gruplarda ham ortalama gürültülü
  (örnek: 97 gruptan 3'ü <50 satır; bir grubun ham ortalamasi 0'dı).
- iki kademeli yedek: il+bolge+guc -> il+guc -> genel ortalama.
- ufuk_gun: modelin kaç gun ileriyi tahmin ettiğini bilmesi için (hata ufukla büyüyor).

In [ ]:
def veri_seti_kur(gecmis, hedef_satirlar, origin_tarihi, k=30):
    g = gecmis[gecmis['tarih'] < origin_tarihi].copy()
    g['log_t']  = np.log1p(g['tuketim_duzeltilmis'])
    genel_log   = g['log_t'].mean()

    grup = g.groupby(['il','bolge','guc_bucket'], observed=True)['log_t'].agg(
        ['sum','count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k * genel_log) / (grup['count'] + k)

    il_guc = (g.groupby(['il','guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index())

    profil   = trafo_profili_olustur(gecmis, origin_tarihi)
    gun_prof = trafo_gun_profili(gecmis, origin_tarihi)
    
    df = hedef_satirlar.drop(columns=['grup_log_ort','ilguc_log_ort'], errors='ignore').copy()
    df = df.merge(grup[['il','bolge','guc_bucket','grup_log_ort']], on=['il','bolge','guc_bucket'], how='left')
    df = df.merge(il_guc, on=['il','guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel_log)

    df = df.merge(profil,   on='tanim', how='left')
    df = df.merge(gun_prof, on=['tanim','haftanin_gunu'], how='left')
    
    mevsim = grup_mevsim_profili(gecmis, origin_tarihi)
    df = df.merge(mevsim, on=['bolge','guc_bucket','ay'], how='left')
    df['grup_ay_sapma'] = df['grup_ay_sapma'].fillna(0.0)
    df['grup_ay_n'] = df['grup_ay_n'].fillna(0)

    df['ufuk_gun'] = (df['tarih'] - origin_tarihi).dt.days
    return df


DIRECT_FEATURES = [
    'guc','ay','gun','haftanin_gunu','hafta_sonu','yilin_gunu','tatil',
    'il','bolge','ilce','guc_bucket_str','grup_log_ort',
    'trafo_ortalama','trafo_medyan','trafo_gun_sayisi',
    'log_ortalama','log_medyan','log_std','sifir_orani',
    'son30_log_ort','son90_log_ort','son30_sifir','son90_sifir',
    'log_trend','trafo_gun_orani',
]
print('Özellik sayısı:', len(DIRECT_FEATURES))

Özellik sayısı: 25


In [76]:
def origin_seti(hedef_bas, hedef_bit, origin):
    return veri_seti_kur(train, train[(train['tarih'] >= hedef_bas) & (train['tarih'] <= hedef_bit)], pd.Timestamp(origin))

# Hedefler 30 Kasım 2025'te bitiyor -> validation dönemine (Aralık-Mart) sızıntı yok
egitim_seti = pd.concat([
    origin_seti('2025-08-01', '2025-11-30', '2025-08-01'),
    origin_seti('2025-09-01', '2025-11-30', '2025-09-01'),
    origin_seti('2025-10-01', '2025-11-30', '2025-10-01'),
], ignore_index=True)

val_seti = origin_seti('2025-12-01', '2026-03-31', '2025-12-01')

for df in [egitim_seti, val_seti]:
    for c in KATEGORIK:
        df[c] = df[c].astype('category')
    df['hedef_log'] = np.log1p(df['tuketim_duzeltilmis'])

print('eğitim', egitim_seti.shape, '| validation', val_seti.shape)
print('validation ufku:', val_seti['ufuk_gun'].max(), 'gun')


eğitim (726337, 36) | validation (444076, 36)
validation ufku: 120 gun


In [77]:
direct_model = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.05, num_leaves=63, random_state=42
)
direct_model.fit(
    egitim_seti[DIRECT_FEATURES], egitim_seti['hedef_log'],
    eval_set=[(val_seti[DIRECT_FEATURES], val_seti['hedef_log'])],
    eval_metric='rmse',
    categorical_feature=KATEGORIK,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
)

val_seti['tahmin'] = np.clip(np.expm1(direct_model.predict(val_seti[DIRECT_FEATURES])), 0, None)
print('RMSLE          :', rmsle(val_seti['tuketim_duzeltilmis'], val_seti['tahmin']))
print('best_iteration :', direct_model.best_iteration_)

_test = val_seti['tahmin'].mask(val_seti['sifir_orani'] >= 0.9, 0.0)


/Users/beratmertkayacan/Desktop/GridUpDatathon/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019450 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3345
[LightGBM] [Info] Number of data points in the train set: 726337, number of used features: 25
[LightGBM] [Info] Start training from score 6.461310
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[90]	valid_0's rmse: 0.987587	valid_0's l2: 0.975328
RMSLE          : 0.987586818820666
best_iteration : 90


In [78]:
onem = pd.DataFrame({'ozellik': DIRECT_FEATURES, 'onem': direct_model.feature_importances_})
print(onem.sort_values('onem', ascending=False).to_string(index=False))

         ozellik  onem
      yilin_gunu   763
            ilce   746
             guc   493
           bolge   467
         log_std   430
   son30_log_ort   405
trafo_gun_sayisi   339
  trafo_ortalama   317
    grup_log_ort   294
    log_ortalama   249
    trafo_medyan   214
 trafo_gun_orani   199
       log_trend   146
  guc_bucket_str   118
     son30_sifir   103
   son90_log_ort    95
     sifir_orani    91
     son90_sifir    31
      log_medyan    29
             gun    21
              ay    19
              il     6
   haftanin_gunu     5
           tatil     0
      hafta_sonu     0


In [79]:
FINAL_ORIGINLER = ['2025-12-01', '2026-01-01', '2026-02-01']

final_egitim = pd.concat(
    [origin_seti(o, '2026-03-31', o) for o in FINAL_ORIGINLER],ignore_index=True)

for c in KATEGORIK:
    final_egitim[c] = final_egitim[c].astype('category')
final_egitim['hedef_log'] = np.log1p(final_egitim['tuketim_duzeltilmis'])

print('final eğitim:', final_egitim.shape)

final_model = lgb.LGBMRegressor(
    n_estimators=100, # validation'da best_iteration=90 bulundu
    learning_rate=0.05,
    num_leaves=63,
    random_state=42
)
final_model.fit(
    final_egitim[DIRECT_FEATURES], final_egitim['hedef_log'],
    categorical_feature=KATEGORIK
)
print('final model eğitildi')

final eğitim: (1007904, 36)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024884 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3417
[LightGBM] [Info] Number of data points in the train set: 1007904, number of used features: 25
[LightGBM] [Info] Start training from score 6.463323
final model eğitildi


In [80]:
test_seti = veri_seti_kur(train, test, pd.Timestamp('2026-04-01'))

# ÖNEMLİ: kategorileri egitim setinden alıp test'e AYNI sırayla uygula
for c in KATEGORIK:
    test_seti[c] = pd.Categorical(test_seti[c], categories=final_egitim[c].cat.categories)

print('test_seti:', test_seti.shape)
print('kategori uyumu:', all(
    list(test_seti[c].cat.categories) == list(final_egitim[c].cat.categories) for c in KATEGORIK
))
print('profilsiz (cold-start) satır oranı:', round(test_seti['log_ortalama'].isna().mean(), 4))

test_seti: (714688, 32)
kategori uyumu: True
profilsiz (cold-start) satır oranı: 0.2216


In [81]:
tahmin_log = final_model.predict(test_seti[DIRECT_FEATURES])
tahmin = np.clip(np.expm1(tahmin_log), 0, None)  

submission = pd.DataFrame({'id': test_seti['id'], 'tuketim': tahmin})

sample_sub = pd.read_csv('../data/raw/sample_submission.csv')
print('satır sayısı eşit mi:', len(submission) == len(sample_sub))
print('id kümeleri aynı mı:', set(submission['id']) == set(sample_sub['id']))
print('NaN var mı:', submission['tuketim'].isna().sum())
print('negatif var mı:', (submission['tuketim'] < 0).sum())
print()
print(submission['tuketim'].describe())

submission.to_csv('../submissions/sub_01_direct_coklu_origin.csv', index=False)
print('kaydedildi')

satır sayısı eşit mi: True
id kümeleri aynı mı: True
NaN var mı: 0
negatif var mı: 0

count    714688.000000
mean       1701.707644
std        3312.109835
min           0.000000
25%         336.680590
50%         954.466953
75%        2180.124243
max       85083.279652
Name: tuketim, dtype: float64
kaydedildi


1- Mevsimsellik doğrudan look up: Temmuz ayında İZMİR>METROPOL bölgesindeki 250-500 kVA trafolar ortalamadan %40 fazla tüketir bilgisi elimizde var. Temmuz 2025 verisinden hesaplanır, Temmuz 2026'ya doğrudan uygulayacağız.

2- Hava durumu: Sıcaklık mevsim, zamandan bağımsız. Temmuz 2025, 2026 aynı sıcaklıkta aynı kapsamda model için. 1 hangi ay sorusu için, 2 o ayki sıcaklık.


revize: Denendi hiçbir şey ifade etmedi shrankage dahi.

In [89]:
# 2. aday: yaz aylarını da goren egitim seti
YAZ_ORIGINLER   = ['2025-04-01', '2025-05-01', '2025-06-01']   # hedefler Nisan-Eylül 2025
KIS_ORIGINLER   = ['2025-12-01', '2026-01-01', '2026-02-01']   # hedefler Aralık-Mart 

egitim_B = pd.concat(
    [origin_seti(o, '2025-09-30', o) for o in YAZ_ORIGINLER] +
    [origin_seti(o, '2026-03-31', o) for o in KIS_ORIGINLER],
    ignore_index=True
)

for c in KATEGORIK:
    egitim_B[c] = egitim_B[c].astype('category')
egitim_B['hedef_log'] = np.log1p(egitim_B['tuketim_duzeltilmis'])

print('egitim_B:', egitim_B.shape)
print('egitim aylari:', sorted(egitim_B['ay'].unique()))
print('test aylari  :', sorted(test_seti['ay'].unique()))
ortak = set(egitim_B['ay'].unique()) & set(test_seti['ay'].unique())
print('ORTAK ay sayisi:', len(ortak), '->', sorted(ortak))

egitim_B: (2096870, 38)
egitim aylari: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(12)]
test aylari  : [np.int32(4), np.int32(5), np.int32(6), np.int32(7)]
ORTAK ay sayisi: 4 -> [np.int32(4), np.int32(5), np.int32(6), np.int32(7)]


In [ ]:
model_B = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, num_leaves=63, random_state=42)
model_B.fit(egitim_B[DIRECT_FEATURES], egitim_B['hedef_log'], categorical_feature=KATEGORIK)

for c in KATEGORIK:
    test_seti[c] = pd.Categorical(test_seti[c], categories=egitim_B[c].cat.categories)

tahmin_B = np.clip(np.expm1(model_B.predict(test_seti[DIRECT_FEATURES])), 0, None)
sub_B = pd.DataFrame({'id': test_seti['id'], 'tuketim': tahmin_B})

print('satır:', len(sub_B), '| NaN:', sub_B['tuketim'].isna().sum())
print(sub_B['tuketim'].describe())
# Aday A (mevcut, 1.13709) ile kıyas
print('A ortalama tahmin:', round(submission['tuketim'].mean(), 1))
print('B ortalama tahmin:', round(sub_B['tuketim'].mean(), 1))

sub_B.to_csv('../submissions/sub_02_yaz_dahil.csv', index=False)

satir: 714688 | NaN: 0
count    714688.000000
mean       1998.199356
std        3878.472446
min           0.000000
25%         383.320171
50%        1185.026003
75%        2637.080487
max      160719.444072
Name: tuketim, dtype: float64

A ortalama tahmin: 1701.7
B ortalama tahmin: 1998.2
